In [1]:
import os
from sentence_transformers import SentenceTransformer
import faiss

In [3]:
documents = [
    "The weather in New York is cloudy.",
    "The secret password is 'BANANA'.", # Our target
    "Apples are usually red or green.",
    "The capital of France is Paris.",
    "Python is a popular programming language.",
    "The moon orbits the Earth.",
    "Deep learning is a subset of AI.",
    "Water boils at 100 degrees Celsius.",
    "The Great Wall of China is visible from space.",
    "Mount Everest is the tallest mountain."
]

In [6]:
# embedding model
embedding_model = SentenceTransformer('Qwen/Qwen3-Embedding-0.6B')

docs_embeddings = embedding_model.encode(documents, convert_to_numpy=True)

docs_embeddings[0]

array([-0.02539867,  0.01325228, -0.00546455, ...,  0.04256301,
        0.03717346,  0.01400935], shape=(1024,), dtype=float32)

In [30]:
dimensions = docs_embeddings.shape[1]

# index for FAISS
index = faiss.IndexFlatL2(dimensions)
index.add(docs_embeddings)

In [31]:
query = "What is the secret password?"

In [34]:
# retrive documents

query_embedding = embedding_model.encode([query])

top_k = 10
distance, indices = index.search(query_embedding, top_k)

print(f"distance : {distance}, indices : {indices}")

distance : [[0.34783643 1.2334532  1.2765858  1.2931278  1.3005309  1.4182941
  1.4508327  1.5350623  1.6011132  1.6109822 ]], indices : [[1 5 0 4 8 6 7 2 3 9]]


In [35]:
top_chunks = [documents[i] for i in indices[0]]
top_chunks

["The secret password is 'BANANA'.",
 'The moon orbits the Earth.',
 'The weather in New York is cloudy.',
 'Python is a popular programming language.',
 'The Great Wall of China is visible from space.',
 'Deep learning is a subset of AI.',
 'Water boils at 100 degrees Celsius.',
 'Apples are usually red or green.',
 'The capital of France is Paris.',
 'Mount Everest is the tallest mountain.']

In [36]:
# Apply Long Context Reordering
from langchain_community.document_transformers import LongContextReorder

reorder = LongContextReorder()
reordered_documents = reorder.transform_documents(top_chunks)

reordered_documents

['The moon orbits the Earth.',
 'Python is a popular programming language.',
 'Deep learning is a subset of AI.',
 'Apples are usually red or green.',
 'Mount Everest is the tallest mountain.',
 'The capital of France is Paris.',
 'Water boils at 100 degrees Celsius.',
 'The Great Wall of China is visible from space.',
 'The weather in New York is cloudy.',
 "The secret password is 'BANANA'."]